# Eval Metrics Part → Second Leg of Hypothesis Tests (**Metrics Engine**)

In **Leg 2**, we stop “creating signals” and start **measuring performance**.

The key idea is simple:

> If Leg 1 tells us **where** a hypothesis applies (events) and **what it predicts**,  
> Leg 2 tells us **how well it actually works**.

- This notebook exists because we want **one consistent evaluation system** for *all* hypotheses, instead of writing different metric code every time. 
- Given an event definition and (optional) predictions, compute **all evaluation metrics** in a standardized way.

You can think of `eval_metrics.ipynb` as a reusable **metrics calculator**.


## The 3-Leg Pipeline (How the whole system is organized)

### **Leg 1 — prediction.ipynb (Event Factory)** 
**Goal:** Create new columns that mark **where** each hypothesis applies and **what the rule predicts** (if applicable).

- We do **NOT** prove anything here.
- We do **NOT** compute p-values or draw conclusions here.
- We only build:
  - **event masks** (which rows are “in the hypothesis world”)
  - **baseline rule predictions** (optional for some hypotheses)

**Output pattern (typical):**
- `is_Hx` → 0/1 mask for “this row is a valid event for Hx”
- `pred_Hx_*` → rule-based prediction (if the hypothesis is directional)
- additional tags (if the hypothesis is grouping / moderator)

Some hypotheses are already fully defined by existing features and labels → they may pass through this leg with minimal work.

### **Leg 2 — eval_metrics.ipynb (Metrics Engine)** --> WE ARE IN HERE !!!
**Goal:** Given an event definition and predictions, compute all **evaluation metrics** consistently.

Think of this as a reusable “calculator”:

- Input: **(mask, y_true, y_pred)**  
- Output: metrics such as:
  - **hit-rate**
  - **precision / recall / F1** (for rule predictions)
  - **p-value** (example: test `hit-rate > 0.5`)
  - **effect size** (example: signed returns, distance reduction)
  - reusable slice logic (IB width, gap alignment, etc.)

### **Leg 3 — hypothesis_tests.ipynb (Report + Decision Layer)**
**Goal:** Present results in a clean, viewer-friendly form and state decisions clearly.

This is where we:
- show tables/figures
- explain measurement choices (especially for “reversion” type ideas)
- decide:
  - **Reject null hypothesis** or
  - **Fail to reject null hypothesis**
based on the metrics and p-values from Leg 2.


## What we feed into the Metrics Engine → Inputs

For any hypothesis (H1–H5), we only need a small set of objects:

- **`mask`**  
  A boolean or 0/1 filter that selects the rows we are evaluating  

- **`y_true`**  
  The true future outcome we want to evaluate against  

- **`y_pred`** *(only if the hypothesis produces a rule prediction)*  
  The rule-based predicted outcome  


## What the Metrics Engine returns → Outputs

Depending on the hypothesis type, `eval_metrics.ipynb` can compute metrics like:

- **Hit-rate (accuracy)**  
  “On the masked event rows, what fraction of predictions were correct?”

- **Precision / Recall / F1** *(for direction/rule hypotheses)*  
  Extra classification quality metrics, especially useful when class balance matters.

- **p-value (statistical significance)**  
  Examples:
  - Test whether **hit-rate > 0.5** (better than random guessing)
  - Test whether **mean return ≠ 0** (or > 0 / < 0 depending on hypothesis)

- **Effect size (economic/statistical magnitude)**  
  Examples:
  - mean/median **signed returns** (`ret15`, `ret30`)
  - “distance reduction” style outcomes for reversion hypotheses  
    (example: did `|close_f15 - ib_mid|` shrink vs now?)

- **Reusable “slice” logic (conditional analysis)**  
  The same hypothesis can be evaluated under conditions such as:
  - **IB width regime:** narrow vs wide (`is_H6_narrow`, `is_H6_wide`)
  - **Gap alignment:** aligned vs not (`is_H7_align`)
  - other sensitivity filters (like whipsaw vs no-whipsaw)

    * This allows clean comparisons like:
      - “Does H2 work better on wide-IB days?”
      - “Does H4 improve when whipsaw is excluded?”
      - “Are returns larger when gap alignment is present?”

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

### **H1: Direction drift in between state**

#### **(a) Mask (Which rows do we evaluate?)**

H1 should **not** be evaluated on every 1-minute candle.  
We only want rows that are:

1. **Actually valid H1 events**, and
2. **Inside our analysis universe**, and
3. **Safe for label creation** (so the “future” values exist and are not broken)

That is why we combine multiple masks:

- **`is_H1`**  
  Created in **Leg 1 (Prediction)**. This is the main H1 event definition.  
  It answers:  
  **“Is this 1-minute candle in the H1 setup world?”**

- **`is_analysis`**  
  A global filter that defines the rows we allow in our statistical evaluation.  
  It answers:  
  **“Is this candle in our official analysis/testing area?”**  
  (Example use: excluding premarket, illiquid sections, or anything outside our study design.)

- **`is_labelwin`**  
  Created in **`labels.ipynb`** to ensure the row is safe for future-label logic.  
  It answers:  
  **“Do we have a reliable future window after this candle to compute 15/30-minute outcomes?”**  
  This prevents broken labels near the end of the day (or missing future bars).

**Practical final mask for H1 evaluation:**
- Evaluate only rows where all conditions hold:
  - `is_H1 == 1`
  - `is_analysis == 1`
  - `is_labelwin == 1`

This guarantees we test H1 only where it is **defined**, **allowed**, and **measurable**.


#### **(b) `y_true` (What actually happened?)**

`y_true` is the **real-world outcome** from the market (our ground truth).

For the directional version of H1, we use:

- **`dir15`** → the true direction **15 minutes later**
- **`dir30`** → the true direction **30 minutes later**

These columns answer:

- “After this candle, did SPY end up **higher (1)** or **lower (0)** after 15/30 minutes?”

So `y_true` is the reality we want to match.


#### **(c) `y_pred` (What does H1 predict should happen?)**

`y_pred` is the output of our hypothesis rule — what H1 *claims* should happen.

For H1, the rule-based baseline predictions (created in Leg 1) are:

- **`pred_H1_dir15`** → H1’s predicted direction for the next 15 minutes
- **`pred_H1_dir30`** → H1’s predicted direction for the next 30 minutes

These represent:

- “In the H1 setup world, price should move **toward the middle**, which we approximate as an up/down prediction.”


In [14]:
PROJECT_ROOT = Path("..").resolve()

DATA_CACHE = PROJECT_ROOT / "data" / "cache"

CACHE_FILE = DATA_CACHE / "spy_1min_et_with_H1_events.csv"

df_eval1 = pd.read_csv(CACHE_FILE, parse_dates=['datetime'])

df_eval1.head()

,datetime,high,low,close,Volume,hl_pct,hl5,hl15,trend_score_m30,ib_high,...,cross_av_od_last5,close_f15,close_f30,ret15,ret30,dir15,dir30,is_H1,pred_H1_dir15,pred_H1_dir30
0,2025-09-08 09:30:00,648.86,648.24,648.260,141588,0.000956,NaN,NaN,NaN,649.06,...,0,648.42,648.24,0.000247,-0.000031,1,0,0,NaN,NaN
1,2025-09-08 09:31:00,648.45,648.15,648.270,42118,0.000463,NaN,NaN,NaN,649.06,...,0,648.28,647.97,0.000015,-0.000463,1,0,0,NaN,NaN
2,2025-09-08 09:32:00,648.46,648.10,648.260,37143,0.000555,NaN,NaN,NaN,649.06,...,0,648.11,648.27,-0.000231,0.000015,0,1,0,NaN,NaN
3,2025-09-08 09:33:00,648.47,648.23,648.400,42231,0.000370,NaN,NaN,NaN,649.06,...,0,648.57,648.24,0.000262,-0.000247,1,0,0,NaN,NaN
4,2025-09-08 09:34:00,648.68,648.32,648.665,23659,0.000555,0.00058,NaN,NaN,649.06,...,0,648.66,648.29,-0.000008,-0.000578,0,0,0,NaN,NaN


In [15]:
# 1) H1 evaluation mask
mask_H1_eval = (
    (df_eval1["is_H1"] == 1) &      
    (df_eval1["is_analysis"] == 1) & 
    (df_eval1["is_labelwin"] == 1)   
)

# 2) Real results (y_true) -> what happened in the market?
y_true_H1_dir15 = df_eval1.loc[mask_H1_eval, "dir15"]
y_true_H1_dir30 = df_eval1.loc[mask_H1_eval, "dir30"]

# 4) H1's prediction (y_pred) -> What should happen in H1's world?
y_pred_H1_dir15 = df_eval1.loc[mask_H1_eval, "pred_H1_dir15"]
y_pred_H1_dir30 = df_eval1.loc[mask_H1_eval, "pred_H1_dir30"]

print("Total rows :", len(df_eval1))
print("H1 eval mask's selections :", mask_H1_eval.sum())

print("First 5 observations for 15 min (y_true vs y_pred):")
print(pd.DataFrame({
    "dir15": y_true_H1_dir15.head(),
    "pred_H1_dir15": y_pred_H1_dir15.head()
}))

print("First 5 observations for 30 min (y_true vs y_pred):")
print(pd.DataFrame({
    "dir30": y_true_H1_dir30.head(),
    "pred_H1_dir30": y_pred_H1_dir30.head()
}))


Total rows : 21450
H1 eval mask's selections : 326
First 5 observations for 15 min (y_true vs y_pred):
     dir15  pred_H1_dir15
134      1            0.0
135      1            0.0
144      0            0.0
153      0            0.0
154      0            0.0
First 5 observations for 30 min (y_true vs y_pred):
     dir30  pred_H1_dir30
134      0            0.0
135      0            0.0
144      0            0.0
153      0            0.0
154      0            0.0


In [16]:
import math

# --------------------------
# 0) Setup
# --------------------------
df = df_eval1 # our code includes lots of pandas operations so we need to represent it shortly

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["date"] = df["datetime"].dt.date

# --------------------------
# 1) Binomial p-value
# --------------------------

def _log_choose(n, k): # in our prediction of sample size n events, k events must be true
    return math.lgamma(n+1) - math.lgamma(k+1) - math.lgamma(n-k+1)

def _log_binom_pmf(k, n, p): # log version of the probability of seeing k events as true with exact X = k
    if p <= 0 or p >= 1: # we don't want to consider log(0) scenarios
        return -math.inf
    return _log_choose(n, k) + k*math.log(p) + (n-k)*math.log(1-p)

def _logsumexp(log_terms): # our probabilities may be too low but I don't want to lose them
    # so I got largest of them and scale everything according to that number
    m = max(log_terms)
    if m == -math.inf:
        return -math.inf
    return m + math.log(sum(math.exp(t-m) for t in log_terms))

def binom_cdf(k, n, p): # The most k true events seen probability with 
    # P(X <= k), also can be considered as "acceptance region"
    if k < 0: return 0.0
    if k >= n: return 1.0
    logs = [_log_binom_pmf(i, n, p) for i in range(0, k+1)]
    return float(math.exp(_logsumexp(logs)))

def binom_sf(k_minus_1, n, p): # right tail p-value (probability) for clearly determining the rejection area
    # very important tool in hypothesis decisions
    # P(X >= k) = 1 - P(X <= k-1)
    return float(1.0 - binom_cdf(k_minus_1, n, p))

# --------------------------
# 2) Main metrics function (hit/precision/recall/F1 + signed return + p-value)
# --------------------------
def metrics_from_series(y_true_s, y_pred_s, ret_s, p0=0.5): # we assigned p0 as 0.5 because we need to understand 
    # whether our hypothesis shows us something better than a coin-flip

    m = (~y_true_s.isna()) & (~y_pred_s.isna()) # from same index clear all NaNs
    yt = y_true_s.loc[m].astype(int).to_numpy()
    yp = y_pred_s.loc[m].astype(int).to_numpy()
    rr = ret_s.loc[m].astype(float).to_numpy()

    # if there is no column to evaluate, we don't need to waste our time
    N = int(len(yt))
    if N == 0:
        return {"N": 0}

    # Hit-rate calculation and evaluate the true predictions
    correct = (yt == yp)
    k = int(correct.sum())
    hit_rate = k / N

    # Confusion matrix with stating prediction vs. reality
    tp = int(((yp==1) & (yt==1)).sum()) # prediction -> up, reality -> up
    fp = int(((yp==1) & (yt==0)).sum()) # prediction -> up, reality -> down
    fn = int(((yp==0) & (yt==1)).sum()) # prediction -> down, reality -> up
    tn = int(((yp==0) & (yt==0)).sum()) # prediction -> down, reality -> down

    precision = tp/(tp+fp) if (tp+fp)>0 else np.nan # how many my predictions of up is really up in reality
    recall    = tp/(tp+fn) if (tp+fn)>0 else np.nan # how many real ups I got from my predictions
    f1 = (2*precision*recall/(precision+recall)) if (not np.isnan(precision) and not np.isnan(recall) and (precision+recall)>0) else np.nan
    # harmonic average of both variables

    # p-values under H0: X ~ Binomial(N, p0), X = #correct
    p_greater = binom_sf(k-1, N, p0)         # P(X >= k), better than 50%?
    p_less    = binom_cdf(k,   N, p0)        # P(X <= k), worse than 50%?
    p_two     = float(min(1.0, 2*min(p_greater, p_less))) # two tailed hypothesis graph with not exceeding 1.

    signed_r = np.where(yp==1, rr, -rr) # consider if our prediction shows up take return as +, if not take return as -
    mean_sr = float(np.nanmean(signed_r)) # if I am opening positions at just prediction's side what would be my mean return
    med_sr  = float(np.nanmedian(signed_r)) # if I am opening positions at just prediction's side what would be my median return

    p_true = float(yt.mean()) # the ratio of "up" values; up means 1, down means 0 we consider both of them
    # if one of them is majority we will measure its direct ratio of that
    majority_acc = float(max(p_true, 1-p_true)) # also we are considering which one is the most successful p_value
    # if up's (1) are majority we take up's, otherwise down's (0)

    return {
        "N": N, "k_correct": k, "hit_rate": hit_rate,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "pval_greater(>p0)": p_greater,
        "pval_less(<p0)": p_less,
        "pval_two_sided": p_two,
        "mean_signed_return": mean_sr,
        "median_signed_return": med_sr,
        "y_true_up_rate": p_true,
        "majority_baseline_acc": majority_acc,
    } # reporting in a dictionary format


# If value returns NaN, directly says "not calculated"
def _fmt(x):
    return "not calculated" if (x is None or (isinstance(x,float) and np.isnan(x))) else x
def pretty(d):
    return {k:_fmt(v) for k,v in d.items()} if isinstance(d, dict) else d

# --------------------------
# 3) Bootstrap (daily basis)
# --------------------------
# In general, the logic is very simple:
# - Selected days in our sample by mask considered as blocks 
# - We calculate bootstrap confidence interval from that.

def bootstrap_by_day(mask, y_true_col, y_pred_col, ret_col, B=2000, seed=7):
    # Just taking our needed day columns, with dropping every NaN columns because we won't need them
    sub = df.loc[mask, ["date", y_true_col, y_pred_col, ret_col]].dropna(subset=[y_true_col, y_pred_col]).copy()
    if sub.empty:
        return None

    # row based numpy operations
    # we convert every needed things as arrays to make our bootstrap operations faster
    day = sub["date"].to_numpy()
    yt  = sub[y_true_col].astype(int).to_numpy()
    yp  = sub[y_pred_col].astype(int).to_numpy()
    rr  = sub[ret_col].astype(float).to_numpy()
    signed_r = np.where(yp==1, rr, -rr)

    # gün bazında topla (bootstrap'ta concat yok)
    days, inv = np.unique(day, return_inverse=True)
    n_days = len(days)

    # outputting daily basis statistics
    # inv -> tells directly which index corresponds to which trading day 
    # np.bincount(inv) -> how many rows available for each day
    N_d  = np.bincount(inv)
    k_d  = np.bincount(inv, weights=(yt==yp).astype(int))
    tp_d = np.bincount(inv, weights=((yp==1)&(yt==1)).astype(int))
    fp_d = np.bincount(inv, weights=((yp==1)&(yt==0)).astype(int))
    fn_d = np.bincount(inv, weights=((yp==0)&(yt==1)).astype(int))
    sr_d = np.bincount(inv, weights=signed_r)
    # Our main logic is calculating everything in intraday and after summing all of them
    # This makes our operation much easier

    # in every bootstrap instance, our code needs to select n_days days from selecting in days for randomness
    rng = np.random.default_rng(seed)
    hit_list, f1_list, msr_list = [], [], []
    f1_nan = 0

    for _ in range(B):
        pick = rng.integers(0, n_days, size=n_days)  # replacement
        N  = int(N_d[pick].sum()) # if there is no row in selected day, our code needs to say directly this is NaN value
        if N == 0:
            hit_list.append(np.nan); f1_list.append(np.nan); msr_list.append(np.nan); f1_nan += 1
            continue

        # the summation of selected days summaries -> the summary of bootstrap world's summation
        k  = float(k_d[pick].sum())
        tp = float(tp_d[pick].sum())
        fp = float(fp_d[pick].sum())
        fn = float(fn_d[pick].sum())
        sr = float(sr_d[pick].sum())

        # saving that bootstrap summation hit-rate
        hit_list.append(k / N)

        # saving that bootstrap summation's F1
        prec = tp/(tp+fp) if (tp+fp)>0 else np.nan
        rec  = tp/(tp+fn) if (tp+fn)>0 else np.nan
        f1 = (2*prec*rec/(prec+rec)) if (not np.isnan(prec) and not np.isnan(rec) and (prec+rec)>0) else np.nan
        if np.isnan(f1): f1_nan += 1
        f1_list.append(f1)

        # when days are resampling, our code needs to save average signed return also
        msr_list.append(sr / N)

    # Now, we are dropping all NaNs and returning real Confidence Interval
    # with 2.5%, 50% and 97.5% quartiles
    def ci(arr):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        if len(arr)==0:
            return (np.nan, np.nan, np.nan)
        return (float(np.quantile(arr,0.025)), float(np.quantile(arr,0.50)), float(np.quantile(arr,0.975)))

    # returning our bootstrap results dictionary
    return {
        "n_days": int(n_days), "B": int(B),
        "hit_ci(2.5,50,97.5)": ci(hit_list),
        "f1_ci(2.5,50,97.5)": ci(f1_list),
        "mean_signed_return_ci(2.5,50,97.5)": ci(msr_list),
        "f1_nan_rate": f1_nan / B
    }

# --------------------------
# 4) RUN (overall + slices + bootstrap) — summary outputs just for visualization and showing the code really works
# --------------------------
out_hypothesis1 = {}

# Overall metrics
out_hypothesis1["overall_15"] = metrics_from_series(y_true_H1_dir15, y_pred_H1_dir15, df.loc[mask_H1_eval, "ret15"], p0=0.5)
out_hypothesis1["overall_30"] = metrics_from_series(y_true_H1_dir30, y_pred_H1_dir30, df.loc[mask_H1_eval, "ret30"], p0=0.5)

# Slice: ib_width_type (narrow/wide) — yine same definition, only mask is filtering
for val in ["narrow", "wide"]:
    m = mask_H1_eval & (df["ib_width_type"] == val)
    out_hypothesis1[f"{val}_15"] = metrics_from_series(df.loc[m,"dir15"], df.loc[m,"pred_H1_dir15"], df.loc[m,"ret15"], p0=0.5)
    out_hypothesis1[f"{val}_30"] = metrics_from_series(df.loc[m,"dir30"], df.loc[m,"pred_H1_dir30"], df.loc[m,"ret30"], p0=0.5)

# Bootstrap (day-by-day)
out_hypothesis1["bootstrap_overall_15"] = bootstrap_by_day(mask_H1_eval, "dir15", "pred_H1_dir15", "ret15", B=2000, seed=7)
out_hypothesis1["bootstrap_overall_30"] = bootstrap_by_day(mask_H1_eval, "dir30", "pred_H1_dir30", "ret30", B=2000, seed=7)

# Print
for k,v in out_hypothesis1.items():
    print("\n====================", k, "====================")
    print(pretty(v))



==================== overall_15 ====================
{'N': 326, 'k_correct': 135, 'hit_rate': 0.41411042944785276, 'tp': 68, 'fp': 111, 'fn': 80, 'tn': 67, 'precision': 0.37988826815642457, 'recall': 0.4594594594594595, 'f1': 0.41590214067278286, 'pval_greater(>p0)': 0.9992225940463264, 'pval_less(<p0)': 0.0011340343275836327, 'pval_two_sided': 0.0022680686551672653, 'mean_signed_return': -0.0002096181345698247, 'median_signed_return': -0.00010388184773285, 'y_true_up_rate': 0.4539877300613497, 'majority_baseline_acc': 0.5460122699386503}

==================== overall_30 ====================
{'N': 326, 'k_correct': 156, 'hit_rate': 0.4785276073619632, 'tp': 89, 'fp': 90, 'fn': 80, 'tn': 67, 'precision': 0.4972067039106145, 'recall': 0.5266272189349113, 'f1': 0.5114942528735632, 'pval_greater(>p0)': 0.7969302581294637, 'pval_less(<p0)': 0.2357889885645186, 'pval_two_sided': 0.4715779771290372, 'mean_signed_return': -0.00011292030256727929, 'median_signed_return': -6.624324343346144e-05

In [17]:
# saving our results in a .json format for easily readable structure into 'reports' folder
# the reason of our save is using these results in 08_hypothesis_tests.ipynb file

import json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
OUT_DIR = PROJECT_ROOT / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSON = OUT_DIR / "H1_metrics.json"

# viewer-friendly payload
payload = {k: pretty(v) for k, v in out_hypothesis1.items()}

# write
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", OUT_JSON)

Saved: /Users/canka/Dev/python/DSA210-Project-Can-Karadogan/reports/H1_metrics.json


### **H2: Continuation when both agree**

H2 is the cleanest “rule-style” hypothesis in the project.

The idea is:

> When price is **clearly above both AVWAPs** or **clearly below both AVWAPs**,  
> **and** the two AVWAP slopes point in the **same direction**,  
> price is more likely than random to **continue** in that direction over the next **15–30 minutes**.


#### **(a) Mask (Which rows do we evaluate?)**

We **do not** evaluate H2 on every 1-minute candle.  
We only evaluate candles that satisfy three conditions:

1. **It is a real H2 event** (H2 logic is active on that row)
2. **It belongs to our analysis universe** (we allow this part of the data)
3. **It is safe for labeling** (the future 15/30-minute outcome exists)

That is why we combine three masks:

- **`is_H2`**  
  Created in **Leg 1 (Prediction)**. This is the official H2 event definition.  
  It answers: **“Is this candle in the H2 setup world?”**

- **`is_analysis`**  
  A global filter that defines which rows are valid in our study design.  
  It answers: **“Is this candle inside our testing area?”**

- **`is_labelwin`**  
  Created in **`labels.ipynb`**. It prevents broken labels near end-of-day.  
  It answers: **“Do we have valid future bars to compute 15/30-minute outcomes?”**


Evaluate only rows where:

- `is_H2 == 1`
- `is_analysis == 1`
- `is_labelwin == 1`

This guarantees we test H2 only where it is **defined**, **allowed**, and **measurable**.


#### **(b) `y_true` (What actually happened?)**

`y_true` is the **real future outcome** from the market (ground truth).

For H2, we mainly care about directional continuation over **30 minutes**, so the best primary label is:

- **`dir30`** → the true direction 30 minutes later  
  (1 = up, 0 = down)

We also use:

- **`dir15`** → the true direction 15 minutes later  
  (robustness check: does it work on a shorter horizon too?)

These answer:
> “After this candle, did SPY end up higher (1) or lower (0) after 15/30 minutes?”

##### **Supporting truth columns (effect size)**
Directional accuracy is “was it right?”, but we also want “how big was it?”.

So we should also report:

- **`ret30`**, **`ret15`** → realized return (%) over 30/15 minutes
- **`close_f30`**, **`close_f15`** → future close price (raw reference)

**Clean H2 reporting:**
- Use **`dir30`** as the main `y_true`
- Show **`ret30`** alongside it as the effect-size companion


#### **(c) `y_pred` (What does H2 predict should happen?)**

`y_pred` is what the hypothesis rule predicted.

- **`pred_H2_dir30`** → H2’s predicted direction for the next 30 minutes (main)
- **`pred_H2_dir15`** → H2’s predicted direction for the next 15 minutes (robustness)

##### **How H2 prediction is determined (conceptually)**
H2’s predicted direction is basically encoded by position:

- if `state_ud_above == 1` → predict **up** (1)
- if `state_ud_below == 1` → predict **down** (0)

And the “both agree” slopes decide whether the event is valid (whether we trust it):

- the slope agreement filter is what makes it **H2**, not just “above/below”.

So H2 is essentially:

> **Only trust continuation when:**
> - price is clearly above/below both anchors **and**
> - both anchors are drifting in the same direction (non-zero agreement)


In [9]:
PROJECT_ROOT = Path("..").resolve()

DATA_CACHE = PROJECT_ROOT / "data" / "cache"

CACHE_FILE = DATA_CACHE / "spy_1min_et_with_H2_events.csv"

df_eval2 = pd.read_csv(CACHE_FILE, parse_dates=['datetime'])

df_eval2.head()

,datetime,high,low,close,Volume,hl_pct,hl5,hl15,trend_score_m30,ib_high,...,cross_av_od_last5,close_f15,close_f30,ret15,ret30,dir15,dir30,is_H2,pred_H2_dir15,pred_H2_dir30
0,2025-09-08 09:30:00,648.86,648.24,648.260,141588,0.000956,NaN,NaN,NaN,649.06,...,0,648.42,648.24,0.000247,-0.000031,1,0,0,NaN,NaN
1,2025-09-08 09:31:00,648.45,648.15,648.270,42118,0.000463,NaN,NaN,NaN,649.06,...,0,648.28,647.97,0.000015,-0.000463,1,0,0,NaN,NaN
2,2025-09-08 09:32:00,648.46,648.10,648.260,37143,0.000555,NaN,NaN,NaN,649.06,...,0,648.11,648.27,-0.000231,0.000015,0,1,0,NaN,NaN
3,2025-09-08 09:33:00,648.47,648.23,648.400,42231,0.000370,NaN,NaN,NaN,649.06,...,0,648.57,648.24,0.000262,-0.000247,1,0,0,NaN,NaN
4,2025-09-08 09:34:00,648.68,648.32,648.665,23659,0.000555,0.00058,NaN,NaN,649.06,...,0,648.66,648.29,-0.000008,-0.000578,0,0,0,NaN,NaN


In [10]:
# ==========================
# H2 (mask, y_true, y_pred)
# ==========================

# 1) H2 evaluation mask
mask_H2_eval = (
    (df_eval2["is_H2"] == 1) &        # real H2 events (Leg 1)
    (df_eval2["is_analysis"] == 1) &  # inside analysis universe
    (df_eval2["is_labelwin"] == 1)    # safe for 15/30m labeling
)

# 2) Real results (y_true) -> what happened in the market?
y_true_H2_dir15 = df_eval2.loc[mask_H2_eval, "dir15"]
y_true_H2_dir30 = df_eval2.loc[mask_H2_eval, "dir30"]

# 3) H2's prediction (y_pred) -> what should happen in H2's world?
y_pred_H2_dir15 = df_eval2.loc[mask_H2_eval, "pred_H2_dir15"]
y_pred_H2_dir30 = df_eval2.loc[mask_H2_eval, "pred_H2_dir30"]

print("Total rows :", len(df_eval2))
print("H2 eval mask's selections :", int(mask_H2_eval.sum()))

print("First 5 observations for 15 min (y_true vs y_pred):")
print(pd.DataFrame({
    "dir15": y_true_H2_dir15.head(),
    "pred_H2_dir15": y_pred_H2_dir15.head()
}))

print("First 5 observations for 30 min (y_true vs y_pred):")
print(pd.DataFrame({
    "dir30": y_true_H2_dir30.head(),
    "pred_H2_dir30": y_pred_H2_dir30.head()
}))


Total rows : 21450
H2 eval mask's selections : 13577
First 5 observations for 15 min (y_true vs y_pred):
    dir15  pred_H2_dir15
75      1            1.0
76      1            1.0
77      1            1.0
78      1            1.0
79      1            1.0
First 5 observations for 30 min (y_true vs y_pred):
    dir30  pred_H2_dir30
75      1            1.0
76      1            1.0
77      0            1.0
78      0            1.0
79      0            1.0


In [11]:
import math

# Same logic repeating here with H1 metrics

# --------------------------
# 0) Setup
# --------------------------
df = df_eval2

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["date"] = df["datetime"].dt.date

# --------------------------
# 1) Binomial p-value (pure Python, numerically safe)
# --------------------------

def _log_choose(n, k):
    return math.lgamma(n+1) - math.lgamma(k+1) - math.lgamma(n-k+1)

def _log_binom_pmf(k, n, p):
    if p <= 0 or p >= 1:
        return -math.inf
    return _log_choose(n, k) + k*math.log(p) + (n-k)*math.log(1-p)

def _logsumexp(log_terms):
    m = max(log_terms)
    if m == -math.inf:
        return -math.inf
    return m + math.log(sum(math.exp(t-m) for t in log_terms))

def binom_cdf(k, n, p):
    # P(X <= k)
    if k < 0: return 0.0
    if k >= n: return 1.0
    logs = [_log_binom_pmf(i, n, p) for i in range(0, k+1)]
    out = float(math.exp(_logsumexp(logs)))
    return float(max(0.0, min(1.0, out)))

def binom_sf(k_minus_1, n, p):
    # P(X >= k) = 1 - P(X <= k-1)
    out = float(1.0 - binom_cdf(k_minus_1, n, p))
    return float(max(0.0, min(1.0, out)))

# --------------------------
# 2) Main metrics function (hit/precision/recall/F1 + signed return + p-value)
# --------------------------
def metrics_from_series(y_true_s, y_pred_s, ret_s, p0=0.5):
    m = (~y_true_s.isna()) & (~y_pred_s.isna())  # index-aligned NaN drop
    yt = y_true_s.loc[m].astype(int).to_numpy()
    yp = y_pred_s.loc[m].astype(int).to_numpy()
    rr = ret_s.loc[m].astype(float).to_numpy()

    N = int(len(yt))
    if N == 0:
        return {"N": 0}

    correct = (yt == yp)
    k = int(correct.sum())
    hit_rate = k / N

    tp = int(((yp==1) & (yt==1)).sum())
    fp = int(((yp==1) & (yt==0)).sum())
    fn = int(((yp==0) & (yt==1)).sum())
    tn = int(((yp==0) & (yt==0)).sum())

    precision = tp/(tp+fp) if (tp+fp)>0 else np.nan
    recall    = tp/(tp+fn) if (tp+fn)>0 else np.nan
    f1 = (2*precision*recall/(precision+recall)) if (not np.isnan(precision) and not np.isnan(recall) and (precision+recall)>0) else np.nan

    # p-values under H0: X ~ Binomial(N, p0), X = #correct
    p_greater = binom_sf(k-1, N, p0)
    p_less    = binom_cdf(k,   N, p0)
    p_two     = float(min(1.0, 2.0*min(p_greater, p_less)))

    p_greater = float(max(0.0, min(1.0, p_greater)))
    p_less    = float(max(0.0, min(1.0, p_less)))
    p_two     = float(max(0.0, min(1.0, p_two)))

    signed_r = np.where(yp==1, rr, -rr)
    mean_sr = float(np.nanmean(signed_r))
    med_sr  = float(np.nanmedian(signed_r))

    p_true = float(yt.mean())
    majority_acc = float(max(p_true, 1-p_true))

    return {
        "N": N, "k_correct": k, "hit_rate": hit_rate,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "pval_greater(>p0)": p_greater,
        "pval_less(<p0)": p_less,
        "pval_two_sided": p_two,
        "mean_signed_return": mean_sr,
        "median_signed_return": med_sr,
        "y_true_up_rate": p_true,
        "majority_baseline_acc": majority_acc,
    }

def _fmt(x):
    return "not calculated" if (x is None or (isinstance(x,float) and np.isnan(x))) else x

def pretty(d):
    return {k:_fmt(v) for k,v in d.items()} if isinstance(d, dict) else d

# --------------------------
# 3) Bootstrap (daily basis)
# --------------------------
def bootstrap_by_day(mask, y_true_col, y_pred_col, ret_col, B=2000, seed=7):
    # dropna includes ret_col too (prevents weights issues)
    sub = df.loc[mask, ["date", y_true_col, y_pred_col, ret_col]].dropna(
        subset=[y_true_col, y_pred_col, ret_col]
    ).copy()
    if sub.empty:
        return None

    day = sub["date"].to_numpy()
    yt  = sub[y_true_col].astype(int).to_numpy()
    yp  = sub[y_pred_col].astype(int).to_numpy()
    rr  = sub[ret_col].astype(float).to_numpy()
    signed_r = np.where(yp==1, rr, -rr)

    days, inv = np.unique(day, return_inverse=True)
    n_days = len(days)

    N_d  = np.bincount(inv)
    k_d  = np.bincount(inv, weights=(yt==yp).astype(int))
    tp_d = np.bincount(inv, weights=((yp==1)&(yt==1)).astype(int))
    fp_d = np.bincount(inv, weights=((yp==1)&(yt==0)).astype(int))
    fn_d = np.bincount(inv, weights=((yp==0)&(yt==1)).astype(int))
    sr_d = np.bincount(inv, weights=signed_r)

    rng = np.random.default_rng(seed)
    hit_list, f1_list, msr_list = [], [], []
    f1_nan = 0

    for _ in range(B):
        pick = rng.integers(0, n_days, size=n_days)  # resample days with replacement
        N  = int(N_d[pick].sum())
        if N == 0:
            hit_list.append(np.nan); f1_list.append(np.nan); msr_list.append(np.nan); f1_nan += 1
            continue

        k  = float(k_d[pick].sum())
        tp = float(tp_d[pick].sum())
        fp = float(fp_d[pick].sum())
        fn = float(fn_d[pick].sum())
        sr = float(sr_d[pick].sum())

        hit_list.append(k / N)

        prec = tp/(tp+fp) if (tp+fp)>0 else np.nan
        rec  = tp/(tp+fn) if (tp+fn)>0 else np.nan
        f1 = (2*prec*rec/(prec+rec)) if (not np.isnan(prec) and not np.isnan(rec) and (prec+rec)>0) else np.nan
        if np.isnan(f1): f1_nan += 1
        f1_list.append(f1)

        msr_list.append(sr / N)

    def ci(arr):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        if len(arr)==0:
            return (np.nan, np.nan, np.nan)
        return (float(np.quantile(arr,0.025)),
                float(np.quantile(arr,0.50)),
                float(np.quantile(arr,0.975)))

    return {
        "n_days": int(n_days), "B": int(B),
        "hit_ci(2.5,50,97.5)": ci(hit_list),
        "f1_ci(2.5,50,97.5)": ci(f1_list),
        "mean_signed_return_ci(2.5,50,97.5)": ci(msr_list),
        "f1_nan_rate": f1_nan / B
    }

# --------------------------
# 4) RUN (overall + H2-aligned slices + bootstrap)
# --------------------------
out_hypothesis2 = {}

# Overall
out_hypothesis2["overall_15"] = metrics_from_series(
    y_true_H2_dir15, y_pred_H2_dir15, df.loc[mask_H2_eval, "ret15"], p0=0.5
)
out_hypothesis2["overall_30"] = metrics_from_series(
    y_true_H2_dir30, y_pred_H2_dir30, df.loc[mask_H2_eval, "ret30"], p0=0.5
)

# Slice A: above vs below (directly matches the hypothesis wording)
for side, col in [("above", "state_ud_above"), ("below", "state_ud_below")]:
    m = mask_H2_eval & (df[col] == 1)
    out_hypothesis2[f"{side}_15"] = metrics_from_series(df.loc[m,"dir15"], df.loc[m,"pred_H2_dir15"], df.loc[m,"ret15"], p0=0.5)
    out_hypothesis2[f"{side}_30"] = metrics_from_series(df.loc[m,"dir30"], df.loc[m,"pred_H2_dir30"], df.loc[m,"ret30"], p0=0.5)

# Slice B: both slopes up vs both slopes down (agreement direction)
for sgn, name in [(1, "slopes_up"), (-1, "slopes_down")]:
    m = mask_H2_eval & (df["slope_up_sign"] == sgn) & (df["slope_down_sign"] == sgn)
    out_hypothesis2[f"{name}_15"] = metrics_from_series(df.loc[m,"dir15"], df.loc[m,"pred_H2_dir15"], df.loc[m,"ret15"], p0=0.5)
    out_hypothesis2[f"{name}_30"] = metrics_from_series(df.loc[m,"dir30"], df.loc[m,"pred_H2_dir30"], df.loc[m,"ret30"], p0=0.5)

# Bootstrap (day-by-day)
out_hypothesis2["bootstrap_overall_15"] = bootstrap_by_day(mask_H2_eval, "dir15", "pred_H2_dir15", "ret15", B=2000, seed=7)
out_hypothesis2["bootstrap_overall_30"] = bootstrap_by_day(mask_H2_eval, "dir30", "pred_H2_dir30", "ret30", B=2000, seed=7)

# Print
for k, v in out_hypothesis2.items():
    print("\n====================", k, "====================")
    print(pretty(v))



==================== overall_15 ====================
{'N': 13577, 'k_correct': 7276, 'hit_rate': 0.5359063121455403, 'tp': 3986, 'fp': 3365, 'fn': 2936, 'tn': 3290, 'precision': 0.5422391511358999, 'recall': 0.5758451314648946, 'f1': 0.5585370980172354, 'pval_greater(>p0)': 0.0, 'pval_less(<p0)': 1.0, 'pval_two_sided': 0.0, 'mean_signed_return': 8.998546202952368e-05, 'median_signed_return': 6.184291898558847e-05, 'y_true_up_rate': 0.5098328054798557, 'majority_baseline_acc': 0.5098328054798557}

==================== overall_30 ====================
{'N': 13577, 'k_correct': 7458, 'hit_rate': 0.549311335346542, 'tp': 4183, 'fp': 3168, 'fn': 2951, 'tn': 3275, 'precision': 0.5690382260916882, 'recall': 0.5863470703672554, 'f1': 0.5775629962029686, 'pval_greater(>p0)': 0.0, 'pval_less(<p0)': 1.0, 'pval_two_sided': 0.0, 'mean_signed_return': 0.00015947274553022587, 'median_signed_return': 0.0001189431897588, 'y_true_up_rate': 0.5254474478898137, 'majority_baseline_acc': 0.5254474478898137}

In [12]:
# saving our results in a .json format for easily readable structure into 'reports' folder
# the reason of our save is using these results in 08_hypothesis_tests.ipynb file

import json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
OUT_DIR = PROJECT_ROOT / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSON = OUT_DIR / "H2_metrics.json"

# viewer-friendly payload
payload = {k: pretty(v) for k, v in out_hypothesis2.items()}

# write
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", OUT_JSON)


Saved: /Users/canka/Dev/python/DSA210-Project-Can-Karadogan/reports/H2_metrics.json


### **H3: Third anchor improves stability** (how we evaluate it in `eval_metrics.ipynb`)

H3 is a **comparison hypothesis**.

It does not simply say “predict up/down”.  
It says something deeper:

> If we add **`avwap_open`** to the pair (**`avwap_up`**, **`avwap_down`**),  
> the decisions become **more stable** and the balance of **hits vs errors** improves  
> compared to using only the two-anchor baseline (H2).

**Intuition (plain language):**
- `avwap_open` is the market’s shared starting reference point at the open.
- Using it as a third anchor can reduce noisy flip-flops and help avoid fragile signals.
- So H3 is essentially: **“Does adding the open anchor improve reliability?”**


#### **(a) Mask (Which rows do we evaluate?)**

We do **not** evaluate H3 on all 1-minute candles.

We only evaluate rows that satisfy **four requirements**:

##### **1) `is_H2` → setup world (comparison universe)**
Because H3 is defined as an improvement over H2, the evaluation universe must be:

- **the H2 world**

So:

- **`is_H2 == 1`** answers:  
  **“Is this candle in the 2-anchor continuation setup universe?”**

This is the foundation of fairness:
- If the world is not H2, then “H3 improves H2” is not a valid comparison.


##### **2) `is_analysis` → analysis universe**
This is the global filter that restricts evaluation to the rows we officially allow.

- **`is_analysis == 1`** answers:  
  **“Is this candle inside our official analysis/testing area?”**


##### **3) `is_labelwin` → label safety**
This prevents broken forward labels (for example near end-of-day).

- **`is_labelwin == 1`** answers:  
  **“Do we have a reliable future window after this candle?”**


##### **4) `pred_H3_dir15` / `pred_H3_dir30` → H3 must speak**
H3 can be **selective**. Sometimes it may choose not to produce a prediction.

So we must evaluate H3 only when prediction exists:

- `pred_H3_dir15` not NaN → H3 gives a 15-minute prediction
- `pred_H3_dir30` not NaN → H3 gives a 30-minute prediction

**Key point:**  
We must define masks **separately** for 15 and 30 minutes, because H3 might speak in one horizon but not the other.


#### **Canonical final mask for H3 evaluation (single correct approach)**

##### **For 15-minute metrics**
Evaluate only rows where:

- `is_H2 == 1`
- `is_analysis == 1`
- `is_labelwin == 1`
- `pred_H3_dir15` is **not NaN**

##### **For 30-minute metrics**
Evaluate only rows where:

- `is_H2 == 1`
- `is_analysis == 1`
- `is_labelwin == 1`
- `pred_H3_dir30` is **not NaN**

**Why this matters (common mistake prevention):**
- Do **not** require both `pred_H3_dir15` and `pred_H3_dir30` at the same time,
  because that would shrink the sample unnecessarily and distort results.


##### **Fairness rule (baseline comparison must use identical rows)**

Because H3 is a “versus H2” claim, the comparison must be row-for-row fair:

- For **15 minutes**:
  - compare `pred_H2_dir15` vs `pred_H3_dir15`
  - using the **15-minute H3 mask**

- For **30 minutes**:
  - compare `pred_H2_dir30` vs `pred_H3_dir30`
  - using the **30-minute H3 mask**

So H2 must be evaluated on **the exact same rows** where H3 is active for that horizon.


#### **(b) `y_true` (What actually happened?)**

`y_true` is the **real future outcome** in the market (ground truth) from the CSV.

##### **Directional ground truth (classification labels)**
- **`dir15`** → true direction 15 minutes later (1 = up, 0 = down)
- **`dir30`** → true direction 30 minutes later (1 = up, 0 = down)

These answer:
> “After this candle, did SPY end up higher (1) or lower (0) after 15/30 minutes?”

##### **Magnitude ground truth (supports the “stability” interpretation)**
- **`ret15`** → realized forward return over 15 minutes
- **`ret30`** → realized forward return over 30 minutes

These help us detect whether stability is coming from:
- avoiding noisy micro-moves, not just flipping the binary label.


#### **(c) `y_pred` (What does H3 predict?)**

`y_pred` is the prediction output created by the H3 rule.

From the CSV:

- **`pred_H3_dir15`** → H3 predicted direction for 15 minutes
- **`pred_H3_dir30`** → H3 predicted direction for 30 minutes

Baseline predictions (2-anchor approach) for the “versus H2” claim:

- **`pred_H2_dir15`**
- **`pred_H2_dir30`**

In [20]:
PROJECT_ROOT = Path("..").resolve()

DATA_CACHE = PROJECT_ROOT / "data" / "cache"

CACHE_FILE = DATA_CACHE / "spy_1min_et_with_H3_events.csv"

df_eval3 = pd.read_csv(CACHE_FILE, parse_dates=['datetime'])

df_eval3.head()

,datetime,high,low,close,Volume,hl_pct,hl5,hl15,trend_score_m30,ib_high,...,ret30,dir15,dir30,is_H2,pred_H2_dir15,pred_H2_dir30,is_H3,pred_H3_dir15,pred_H3_dir30,is_H3_strict
0,2025-09-08 09:30:00,648.86,648.24,648.260,141588,0.000956,NaN,NaN,NaN,649.06,...,-0.000031,1,0,0,NaN,NaN,0,NaN,NaN,0
1,2025-09-08 09:31:00,648.45,648.15,648.270,42118,0.000463,NaN,NaN,NaN,649.06,...,-0.000463,1,0,0,NaN,NaN,0,NaN,NaN,0
2,2025-09-08 09:32:00,648.46,648.10,648.260,37143,0.000555,NaN,NaN,NaN,649.06,...,0.000015,0,1,0,NaN,NaN,0,NaN,NaN,0
3,2025-09-08 09:33:00,648.47,648.23,648.400,42231,0.000370,NaN,NaN,NaN,649.06,...,-0.000247,1,0,0,NaN,NaN,0,NaN,NaN,0
4,2025-09-08 09:34:00,648.68,648.32,648.665,23659,0.000555,0.00058,NaN,NaN,649.06,...,-0.000578,0,0,0,NaN,NaN,0,NaN,NaN,0


In [21]:
import pandas as pd

# ============================================
# H3 evaluation inputs (mask, y_true, y_pred)
# ============================================

# 1) Base universe (fair setup + analysis + label safety)
base_universe = (
    (df_eval3["is_H2"] == 1) &
    (df_eval3["is_analysis"] == 1) &
    (df_eval3["is_labelwin"] == 1)
)

# 2) Canonical final masks (H3 must speak, separately per horizon)
#    + guardrail: H2 baseline must also exist on the same rows (row-for-row fair comparison)
mask_H3_eval_15 = (
    base_universe &
    df_eval3["pred_H3_dir15"].notna() &
    df_eval3["pred_H2_dir15"].notna()
)

mask_H3_eval_30 = (
    base_universe &
    df_eval3["pred_H3_dir30"].notna() &
    df_eval3["pred_H2_dir30"].notna()
)

# 3) y_true (ground truth)
y_true_H3_dir15 = df_eval3.loc[mask_H3_eval_15, "dir15"]
y_true_H3_dir30 = df_eval3.loc[mask_H3_eval_30, "dir30"]

#  magnitude ground truth for stability interpretation
y_true_H3_ret15 = df_eval3.loc[mask_H3_eval_15, "ret15"]
y_true_H3_ret30 = df_eval3.loc[mask_H3_eval_30, "ret30"]

# 4) y_pred (H3 predictions)
y_pred_H3_dir15 = df_eval3.loc[mask_H3_eval_15, "pred_H3_dir15"]
y_pred_H3_dir30 = df_eval3.loc[mask_H3_eval_30, "pred_H3_dir30"]

# 5) Fairness rule: baseline H2 predictions on the exact same rows
y_pred_H2_on_H3mask_dir15 = df_eval3.loc[mask_H3_eval_15, "pred_H2_dir15"]
y_pred_H2_on_H3mask_dir30 = df_eval3.loc[mask_H3_eval_30, "pred_H2_dir30"]

print("Total rows :", len(df_eval3))
print("H3 eval mask selections (15m):", int(mask_H3_eval_15.sum()))
print("H3 eval mask selections (30m):", int(mask_H3_eval_30.sum()))

print("\nFirst 5 observations for 15 min (y_true vs y_pred_H3 vs y_pred_H2 on same rows):")
print(pd.DataFrame({
    "dir15": y_true_H3_dir15.head(),
    "pred_H3_dir15": y_pred_H3_dir15.head(),
    "pred_H2_dir15": y_pred_H2_on_H3mask_dir15.head(),
    "ret15": y_true_H3_ret15.head()
}))

print("\nFirst 5 observations for 30 min (y_true vs y_pred_H3 vs y_pred_H2 on same rows):")
print(pd.DataFrame({
    "dir30": y_true_H3_dir30.head(),
    "pred_H3_dir30": y_pred_H3_dir30.head(),
    "pred_H2_dir30": y_pred_H2_on_H3mask_dir30.head(),
    "ret30": y_true_H3_ret30.head()
}))


Total rows : 21450
H3 eval mask selections (15m): 11998
H3 eval mask selections (30m): 11998

First 5 observations for 15 min (y_true vs y_pred_H3 vs y_pred_H2 on same rows):
    dir15  pred_H3_dir15  pred_H2_dir15     ret15
75      1            1.0            1.0  0.000462
76      1            1.0            1.0  0.000123
77      1            1.0            1.0  0.000077
78      1            1.0            1.0  0.000185
79      1            1.0            1.0  0.000223

First 5 observations for 30 min (y_true vs y_pred_H3 vs y_pred_H2 on same rows):
    dir30  pred_H3_dir30  pred_H2_dir30     ret30
75      1            1.0            1.0  0.000370
76      1            1.0            1.0  0.000216
77      0            1.0            1.0 -0.000015
78      0            1.0            1.0 -0.000385
79      0            1.0            1.0 -0.000285


In [ ]:
import math

# --------------------------
# 0) Setup
# --------------------------
df = df_eval3  # keep short name

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["date"] = df["datetime"].dt.date

# --------------------------
# 1) Binomial p-value (same infra)
# --------------------------
def _log_choose(n, k):
    return math.lgamma(n+1) - math.lgamma(k+1) - math.lgamma(n-k+1)

def _log_binom_pmf(k, n, p):
    if p <= 0 or p >= 1:
        return -math.inf
    return _log_choose(n, k) + k*math.log(p) + (n-k)*math.log(1-p)

def _logsumexp(log_terms):
    m = max(log_terms)
    if m == -math.inf:
        return -math.inf
    return m + math.log(sum(math.exp(t-m) for t in log_terms))

def binom_cdf(k, n, p):
    if k < 0: return 0.0
    if k >= n: return 1.0
    logs = [_log_binom_pmf(i, n, p) for i in range(0, k+1)]
    return float(math.exp(_logsumexp(logs)))

def binom_sf(k_minus_1, n, p):
    return float(1.0 - binom_cdf(k_minus_1, n, p))

# --------------------------
# 2) Main metrics function (+ balanced acc, FPR/FNR; matches H3 doc)
# --------------------------
def metrics_from_series(y_true_s, y_pred_s, ret_s, p0=0.5):
    m = (~y_true_s.isna()) & (~y_pred_s.isna()) & (~ret_s.isna())
    yt = y_true_s.loc[m].astype(int).to_numpy()
    yp = y_pred_s.loc[m].astype(int).to_numpy()
    rr = ret_s.loc[m].astype(float).to_numpy()

    N = int(len(yt))
    if N == 0:
        return {"N": 0}

    correct = (yt == yp)
    k = int(correct.sum())
    hit_rate = k / N

    tp = int(((yp==1) & (yt==1)).sum())
    fp = int(((yp==1) & (yt==0)).sum())
    fn = int(((yp==0) & (yt==1)).sum())
    tn = int(((yp==0) & (yt==0)).sum())

    precision = tp/(tp+fp) if (tp+fp)>0 else np.nan
    recall    = tp/(tp+fn) if (tp+fn)>0 else np.nan
    f1 = (2*precision*recall/(precision+recall)) if (not np.isnan(precision) and not np.isnan(recall) and (precision+recall)>0) else np.nan

    # Stability-relevant rates
    tpr = recall
    tnr = tn/(tn+fp) if (tn+fp)>0 else np.nan
    fpr = fp/(fp+tn) if (fp+tn)>0 else np.nan
    fnr = fn/(fn+tp) if (fn+tp)>0 else np.nan
    bal_acc = (tpr + tnr)/2 if (not np.isnan(tpr) and not np.isnan(tnr)) else np.nan

    # p-values under H0: X ~ Binomial(N, p0), X = #correct
    p_greater = binom_sf(k-1, N, p0)
    p_less    = binom_cdf(k,   N, p0)
    p_two     = float(min(1.0, 2*min(p_greater, p_less)))

    signed_r = np.where(yp==1, rr, -rr)
    mean_sr = float(np.nanmean(signed_r))
    med_sr  = float(np.nanmedian(signed_r))

    p_true = float(yt.mean())
    majority_acc = float(max(p_true, 1-p_true))

    return {
        "N": N, "k_correct": k, "hit_rate": hit_rate,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "tpr(up_recall)": tpr, "tnr(down_recall)": tnr,
        "fpr(false_alarm_rate)": fpr, "fnr(miss_rate)": fnr,
        "balanced_accuracy": bal_acc,
        "pval_greater(>p0)": p_greater,
        "pval_less(<p0)": p_less,
        "pval_two_sided": p_two,
        "mean_signed_return": mean_sr,
        "median_signed_return": med_sr,
        "y_true_up_rate": p_true,
        "majority_baseline_acc": majority_acc,
    }

# Error-quality block from the doc: compare |ret| on correct vs incorrect
def error_quality(y_true_s, y_pred_s, ret_s):
    m = (~y_true_s.isna()) & (~y_pred_s.isna()) & (~ret_s.isna())
    if m.sum() == 0:
        return {"N": 0}

    yt = y_true_s.loc[m].astype(int).to_numpy()
    yp = y_pred_s.loc[m].astype(int).to_numpy()
    rr = ret_s.loc[m].astype(float).to_numpy()

    correct = (yt == yp)
    wrong = ~correct
    abs_ret = np.abs(rr)

    return {
        "N": int(len(yt)),
        "N_correct": int(correct.sum()),
        "N_wrong": int(wrong.sum()),
        "abs_ret_mean_correct": float(np.nanmean(abs_ret[correct])) if correct.any() else np.nan,
        "abs_ret_median_correct": float(np.nanmedian(abs_ret[correct])) if correct.any() else np.nan,
        "abs_ret_mean_wrong": float(np.nanmean(abs_ret[wrong])) if wrong.any() else np.nan,
        "abs_ret_median_wrong": float(np.nanmedian(abs_ret[wrong])) if wrong.any() else np.nan,
    }

# If value returns NaN, print "not calculated"
def _fmt(x):
    return "not calculated" if (x is None or (isinstance(x,float) and np.isnan(x))) else x
def pretty(d):
    return {k:_fmt(v) for k,v in d.items()} if isinstance(d, dict) else d

# --------------------------
# 3) Bootstrap (daily basis) — same function, H3 masks
# --------------------------
def bootstrap_by_day(mask, y_true_col, y_pred_col, ret_col, B=2000, seed=7):
    sub = df.loc[mask, ["date", y_true_col, y_pred_col, ret_col]].dropna(subset=[y_true_col, y_pred_col, ret_col]).copy()
    if sub.empty:
        return None

    day = sub["date"].to_numpy()
    yt  = sub[y_true_col].astype(int).to_numpy()
    yp  = sub[y_pred_col].astype(int).to_numpy()
    rr  = sub[ret_col].astype(float).to_numpy()
    signed_r = np.where(yp==1, rr, -rr)

    days, inv = np.unique(day, return_inverse=True)
    n_days = len(days)

    N_d  = np.bincount(inv)
    k_d  = np.bincount(inv, weights=(yt==yp).astype(int))
    tp_d = np.bincount(inv, weights=((yp==1)&(yt==1)).astype(int))
    fp_d = np.bincount(inv, weights=((yp==1)&(yt==0)).astype(int))
    fn_d = np.bincount(inv, weights=((yp==0)&(yt==1)).astype(int))
    tn_d = np.bincount(inv, weights=((yp==0)&(yt==0)).astype(int))
    sr_d = np.bincount(inv, weights=signed_r)

    rng = np.random.default_rng(seed)
    hit_list, f1_list, msr_list, bal_list = [], [], [], []
    f1_nan = 0
    bal_nan = 0

    for _ in range(B):
        pick = rng.integers(0, n_days, size=n_days)

        N = int(N_d[pick].sum())
        if N == 0:
            hit_list.append(np.nan); f1_list.append(np.nan); msr_list.append(np.nan); bal_list.append(np.nan)
            f1_nan += 1; bal_nan += 1
            continue

        k  = float(k_d[pick].sum())
        tp = float(tp_d[pick].sum())
        fp = float(fp_d[pick].sum())
        fn = float(fn_d[pick].sum())
        tn = float(tn_d[pick].sum())
        sr = float(sr_d[pick].sum())

        hit_list.append(k / N)

        prec = tp/(tp+fp) if (tp+fp)>0 else np.nan
        rec  = tp/(tp+fn) if (tp+fn)>0 else np.nan
        f1 = (2*prec*rec/(prec+rec)) if (not np.isnan(prec) and not np.isnan(rec) and (prec+rec)>0) else np.nan
        if np.isnan(f1): f1_nan += 1
        f1_list.append(f1)

        msr_list.append(sr / N)

        tpr = rec
        tnr = tn/(tn+fp) if (tn+fp)>0 else np.nan
        bal = (tpr + tnr)/2 if (not np.isnan(tpr) and not np.isnan(tnr)) else np.nan
        if np.isnan(bal): bal_nan += 1
        bal_list.append(bal)

    def ci(arr):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        if len(arr)==0:
            return (np.nan, np.nan, np.nan)
        return (float(np.quantile(arr,0.025)), float(np.quantile(arr,0.50)), float(np.quantile(arr,0.975)))

    return {
        "n_days": int(n_days), "B": int(B),
        "hit_ci(2.5,50,97.5)": ci(hit_list),
        "f1_ci(2.5,50,97.5)": ci(f1_list),
        "balanced_acc_ci(2.5,50,97.5)": ci(bal_list),
        "mean_signed_return_ci(2.5,50,97.5)": ci(msr_list),
        "f1_nan_rate": f1_nan / B,
        "balanced_acc_nan_rate": bal_nan / B,
    }

# --------------------------
# 4) RUN (overall + error-quality + bootstrap) — H3 vs H2 on same rows
# --------------------------
out_hypothesis3 = {}

# ---- Coverage / Abstain ----
base_universe = (
    (df["is_H2"] == 1) &
    (df["is_analysis"] == 1) &
    (df["is_labelwin"] == 1)
)

baseN = int(base_universe.sum())
cov15 = float(df.loc[base_universe, "pred_H3_dir15"].notna().mean()) if baseN > 0 else np.nan
cov30 = float(df.loc[base_universe, "pred_H3_dir30"].notna().mean()) if baseN > 0 else np.nan

out_hypothesis3["coverage_abstain"] = {
    "base_rows(is_H2&analysis&labelwin)": baseN,
    "coverage_15": cov15,
    "abstain_15": (1.0 - cov15) if not np.isnan(cov15) else np.nan,
    "coverage_30": cov30,
    "abstain_30": (1.0 - cov30) if not np.isnan(cov30) else np.nan,
}

# Overall metrics (H3)
out_hypothesis3["H3_overall_15"] = metrics_from_series(y_true_H3_dir15, y_pred_H3_dir15, y_true_H3_ret15, p0=0.5)
out_hypothesis3["H3_overall_30"] = metrics_from_series(y_true_H3_dir30, y_pred_H3_dir30, y_true_H3_ret30, p0=0.5)

# Baseline metrics (H2) on the exact same rows (fairness)
out_hypothesis3["H2_on_H3mask_15"] = metrics_from_series(y_true_H3_dir15, y_pred_H2_on_H3mask_dir15, y_true_H3_ret15, p0=0.5)
out_hypothesis3["H2_on_H3mask_30"] = metrics_from_series(y_true_H3_dir30, y_pred_H2_on_H3mask_dir30, y_true_H3_ret30, p0=0.5)

# Error quality using realized returns (|ret| on correct vs incorrect)
out_hypothesis3["error_quality_H3_15"] = error_quality(y_true_H3_dir15, y_pred_H3_dir15, y_true_H3_ret15)
out_hypothesis3["error_quality_H3_30"] = error_quality(y_true_H3_dir30, y_pred_H3_dir30, y_true_H3_ret30)
out_hypothesis3["error_quality_H2_on_H3mask_15"] = error_quality(y_true_H3_dir15, y_pred_H2_on_H3mask_dir15, y_true_H3_ret15)
out_hypothesis3["error_quality_H2_on_H3mask_30"] = error_quality(y_true_H3_dir30, y_pred_H2_on_H3mask_dir30, y_true_H3_ret30)

# Bootstrap (day-by-day) on canonical masks
out_hypothesis3["bootstrap_H3_overall_15"] = bootstrap_by_day(mask_H3_eval_15, "dir15", "pred_H3_dir15", "ret15", B=2000, seed=7)
out_hypothesis3["bootstrap_H3_overall_30"] = bootstrap_by_day(mask_H3_eval_30, "dir30", "pred_H3_dir30", "ret30", B=2000, seed=7)

# Bootstrap baseline (H2) on the same canonical masks
out_hypothesis3["bootstrap_H2_on_H3mask_15"] = bootstrap_by_day(mask_H3_eval_15, "dir15", "pred_H2_dir15", "ret15", B=2000, seed=7)
out_hypothesis3["bootstrap_H2_on_H3mask_30"] = bootstrap_by_day(mask_H3_eval_30, "dir30", "pred_H2_dir30", "ret30", B=2000, seed=7)

# Print
for k, v in out_hypothesis3.items():
    print("\n====================", k, "====================")
    print(pretty(v))
